**Tools are functions that an agent can call to perform a specific action.**

**1.BUILT-IN TOOLS**

<div align="center">
    <img src="images/Built-in_tools.png" width="350" height="2000">
</div>

**FileReadTool — Answer from a text file**

In [ ]:
from crewai import Agent, LLM
from crewai_tools import FileReadTool

# File path
file_path = "knowledge/company_policy.txt"

# File reading tool
file_tool = FileReadTool(file_path=file_path)

# Agent
agent = Agent(
    role="File Assistant",
    goal="Answer questions using the provided company policy file",
    backstory="You read company policy files and give clear answers.",
    tools=[file_tool],
    llm=LLM(model="gpt-4o-mini", temperature=0),
    verbose=False
)

result = await agent.kickoff_async("What are the company's working hours?.")
print(result)

**PDFsearchTool - Answer from a PDF**

In [ ]:
from crewai import Agent, Task, Crew, LLM
from crewai_tools import SerperDevTool

question = input("What would you like to search for? ").strip()

agent = Agent(
    role="Web Researcher",
    goal="Find information on the web",
    backstory="You search the web and summarize useful results.",
    tools=[SerperDevTool()],
    llm=LLM(model="gpt-4o-mini", temperature=0),
    verbose=False
)

task = Task(
    description=f"Use the search tool to answer: {question}",
    expected_output="A short answer with a source link.",
    agent=agent
)

crew = Crew(agents=[agent], tasks=[task], verbose=False)
result = await crew.kickoff_async()
print(result.raw)

**Code execution — current replacement for CodeInterpreterTool**

In [ ]:
from crewai import Agent, Task, Crew, LLM
from crewai_tools import E2BPythonTool

agent = Agent(
    role="Python Calculator",
    goal="Use Python to calculate correct answers",
    backstory="You write and run small Python programs.",
    tools=[E2BPythonTool()],
    llm=LLM(model="gpt-4o-mini", temperature=0),
    verbose=False
)

task = Task(
    description="Use the Python tool to calculate the average of 12, 18, and 24.",
    expected_output="The calculated average as one short sentence.",
    agent=agent
)

crew = Crew(agents=[agent], tasks=[task], verbose=False)
result = await crew.kickoff_async()
print(result)

**RAG TOOL - Add a file, then ask about it**

In [ ]:
from crewai import Agent, Task, Crew, LLM
from crewai_tools import RagTool

file_path = "knowledge/company_policy.txt"

rag_tool = RagTool(
    config={
        "embedding_model": {
            "provider": "openai",
            "config": {"model_name": "text-embedding-3-small"}
        }
    }
)
rag_tool.add(data_type="file", path=file_path)

agent = Agent(
    role="Document Researcher",
    goal="Answer questions using the added document",
    backstory="You retrieve relevant passages before answering.",
    tools=[rag_tool],
    llm=LLM(model="gpt-4o-mini", temperature=0),
    verbose=False
)

task = Task(
    description="Use the RAG tool to answer: {question}",
    expected_output="A short answer based on the document.",
    agent=agent
)

crew = Crew(agents=[agent], tasks=[task], verbose=False)

result = await crew.kickoff_async(inputs={"question": "What is the policy of Remote Work?."})
print(result)

**Composio — Connect a GitHub tool to one agent**

In [1]:
from crewai import Agent, Task, Crew, LLM
from composio import Composio
from composio_crewai import CrewAIProvider
from composio.core.models import Toolkits
# Initialize Composio with CrewAI provider
composio = Composio(
    api_key="ck_obX5-8DRl1lPPm6qsEp3",  
    provider=CrewAIProvider()
)

# Create session and get GitHub tools
session = composio.create(
    user_id="vidhya_001",
    toolkits=["github"]
)
print("Composio session created successfully")
github_tools = session.tools()

# Create CrewAI agent
agent = Agent(
    role="GitHub Assistant",
    goal="Answer questions about connected GitHub repositories",
    backstory="You use GitHub tools to retrieve repository information.",
    tools=github_tools,
    llm=LLM(
        model="gpt-4o-mini",
        temperature=0
    ),
    verbose=False
)

# Create task
task = Task(
    description="""
    Use the GitHub tools to list up to three repositories
    accessible to the connected user.
    Do not modify anything.
    """,
    expected_output="""
    Up to three repository names, or a clear explanation
    if GitHub is not connected.
    """,
    agent=agent
)

# Create Crew
crew = Crew(
    agents=[agent],
    tasks=[task],
    verbose=False
)

# Run CrewAI
result = await crew.kickoff_async()

print(result)

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API key: ck_**sEp3', 'code': 801, 'slug': 'APIKey_InvalidAPIKey', 'status': 401, 'request_id': 'cf1557da525040e0a0cf4eea28eaee1f', 'suggested_fix': 'Please check you are using a valid API key.'}}

**2. CUSTOM TOOL - A custom tool in CrewAI is a function you create and give to an agent so it can perform a specific action**

In [ ]:
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool

# Create a custom tool
@tool("Add Numbers")
def add_numbers(first: int, second: int) -> str:
    """Add two whole numbers and return their sum."""
    return str(first + second)
    
first = int(input("Enter the first number: "))
second = int(input("Enter the second number: "))

agent = Agent(
    role="Math Assistant",
    goal="Calculate sums using the Add Numbers tool",
    backstory="You help users with simple calculations.",
    tools=[add_numbers],
    llm=LLM(model="gpt-4o-mini", temperature=0),
    allow_delegation=False,
    verbose=False
)

task = Task(
    description=f"Use the Add Numbers tool to add {first} and {second}.",
    expected_output="The sum as a single number.",
    agent=agent
)

crew = Crew(agents=[agent], tasks=[task], verbose=False)

result = await crew.kickoff_async()
print(result.raw)